[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/35_bpe_solution.ipynb)

# ✅ Solution: Byte-Pair Encoding (BPE)

Implement a simple **BPE tokenizer** — the foundation of GPT/LLaMA tokenization.

### Signature
```python
class SimpleBPE:
    def __init__(self): ...
    def train(self, corpus: list[str], num_merges: int): ...
    def encode(self, text: str) -> list[str]: ...
```

### Algorithm (training)
1. Split each word into characters + `</w>` end marker
2. Count all adjacent pairs across the corpus
3. Merge the most frequent pair into a single token
4. Repeat for `num_merges` iterations


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import math


In [ ]:
# ✅ SOLUTION

class SimpleBPE:
    def __init__(self): self.merges=[]
    def train(self,corpus,num_merges):
        vocab={}
        for word in corpus: vocab[tuple(word)+('</w>',)]=vocab.get(tuple(word)+('</w>',),0)+1
        self.merges=[]
        for _ in range(num_merges):
            pairs={}
            for word,freq in vocab.items():
                for i in range(len(word)-1): pairs[(word[i],word[i+1])]=pairs.get((word[i],word[i+1]),0)+freq
            if not pairs: break
            best=max(pairs,key=pairs.get); self.merges.append(best); new={}
            for word,freq in vocab.items():
                result=[]; i=0
                while i<len(word):
                    if i+1<len(word) and (word[i],word[i+1])==best: result.append(word[i]+word[i+1]); i+=2
                    else: result.append(word[i]); i+=1
                new[tuple(result)]=freq
            vocab=new
    def encode(self,text):
        result=[]
        for word in text.split():
            symbols=list(word)+['</w>']
            for a,b in self.merges:
                i=0
                while i<len(symbols)-1:
                    if symbols[i:i+2]==[a,b]: symbols[i:i+2]=[a+b]
                    else: i+=1
            result.extend(symbols)
        return result


In [ ]:
# Verify
print(SimpleBPE)


In [ ]:
from jax_judge import check
check("bpe")
